# Image Clustering with ImageBind LLM Embeddings

## Assignment (h): Image Clustering using ImageBind Embeddings

**Author:** Nitish  
**Date:** December 2024

---

## Table of Contents
1. Introduction
2. Setup and Installation
3. Load Image Dataset
4. Extract Image Embeddings
5. Clustering Analysis
6. Visualization
7. Evaluation Metrics
8. Conclusion

In [ ]:
!pip install torch torchvision numpy pandas matplotlib seaborn scikit-learn umap-learn Pillow -q

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, calinski_harabasz_score, davies_bouldin_score
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import torch
import torchvision
from torchvision import transforms, datasets, models
from PIL import Image
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. Load Pre-trained Vision Model for Embeddings

We'll use ResNet50 pretrained on ImageNet as our embedding model. For ImageBind-style multimodal embeddings, we extract features from the penultimate layer.

In [ ]:
# Load pretrained ResNet50 model
resnet = models.resnet50(pretrained=True)
resnet.eval()
resnet = resnet.to(device)

# Remove the final classification layer to get embeddings
embedding_model = torch.nn.Sequential(*list(resnet.children())[:-1])
embedding_model.eval()
print("ResNet50 embedding model loaded!")

In [ ]:
# Image preprocessing
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

## 2. Load CIFAR-10 Dataset

In [ ]:
# Load CIFAR-10 dataset
cifar_transform = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

cifar_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=cifar_transform)
class_names = cifar_dataset.classes
print(f"CIFAR-10 classes: {class_names}")
print(f"Dataset size: {len(cifar_dataset)}")

In [ ]:
# Sample subset for faster processing
n_samples = 1000
indices = np.random.choice(len(cifar_dataset), n_samples, replace=False)
subset = torch.utils.data.Subset(cifar_dataset, indices)
dataloader = torch.utils.data.DataLoader(subset, batch_size=32, shuffle=False)

# Get true labels
true_labels = np.array([cifar_dataset.targets[i] for i in indices])
print(f"Sampled {n_samples} images")

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for i, cls in enumerate(class_names):
    idx = np.where(true_labels == i)[0][0]
    img, _ = cifar_dataset[indices[idx]]
    img = img.permute(1, 2, 0).numpy()
    img = (img - img.min()) / (img.max() - img.min())
    ax = axes[i // 5, i % 5]
    ax.imshow(img)
    ax.set_title(cls)
    ax.axis('off')
plt.suptitle('CIFAR-10 Sample Images', fontsize=14)
plt.tight_layout(); plt.show()

## 3. Extract Image Embeddings

In [ ]:
# Extract embeddings
embeddings = []
with torch.no_grad():
    for images, _ in dataloader:
        images = images.to(device)
        features = embedding_model(images)
        features = features.squeeze().cpu().numpy()
        embeddings.append(features)

embeddings = np.vstack(embeddings)
print(f"Embeddings shape: {embeddings.shape}")

In [ ]:
# Normalize embeddings
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings)
print(f"Scaled embeddings shape: {embeddings_scaled.shape}")

## 4. Dimensionality Reduction

In [ ]:
# PCA for visualization
pca = PCA(n_components=50)
embeddings_pca = pca.fit_transform(embeddings_scaled)
print(f"PCA explained variance: {sum(pca.explained_variance_ratio_[:50]):.2%}")

In [ ]:
# UMAP for 2D visualization
import umap
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
embeddings_2d = reducer.fit_transform(embeddings_pca)
print(f"UMAP embeddings: {embeddings_2d.shape}")

In [ ]:
# Visualize embeddings by true class
plt.figure(figsize=(12, 10))
scatter = plt.scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=true_labels, cmap='tab10', s=20, alpha=0.6)
plt.colorbar(scatter, label='Class')
plt.title('Image Embeddings (UMAP) - True Labels', fontsize=14)
plt.xlabel('UMAP 1'); plt.ylabel('UMAP 2')

# Add class labels
for i, name in enumerate(class_names):
    mask = true_labels == i
    center = embeddings_2d[mask].mean(axis=0)
    plt.annotate(name, center, fontsize=10, fontweight='bold', ha='center')
plt.tight_layout(); plt.show()

## 5. K-Means Clustering

In [ ]:
# Find optimal K
inertias, silhouettes = [], []
K_range = range(2, 15)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(embeddings_pca)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(embeddings_pca, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(K_range, inertias, 'bo-'); axes[0].set_xlabel('K'); axes[0].set_ylabel('Inertia')
axes[0].set_title('Elbow Method'); axes[0].axvline(x=10, color='r', linestyle='--')
axes[1].plot(K_range, silhouettes, 'go-'); axes[1].set_xlabel('K'); axes[1].set_ylabel('Silhouette')
axes[1].set_title('Silhouette Score'); axes[1].axvline(x=10, color='r', linestyle='--')
plt.tight_layout(); plt.show()

In [ ]:
# Apply K-Means with K=10
kmeans = KMeans(n_clusters=10, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(embeddings_pca)

print("K-Means Clustering Results (K=10):")
print(f"  Silhouette Score: {silhouette_score(embeddings_pca, cluster_labels):.4f}")
print(f"  Calinski-Harabasz: {calinski_harabasz_score(embeddings_pca, cluster_labels):.4f}")
print(f"  Davies-Bouldin: {davies_bouldin_score(embeddings_pca, cluster_labels):.4f}")
print(f"  ARI: {adjusted_rand_score(true_labels, cluster_labels):.4f}")
print(f"  NMI: {normalized_mutual_info_score(true_labels, cluster_labels):.4f}")

In [ ]:
# Visualize clusters
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

scatter1 = axes[0].scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=true_labels, cmap='tab10', s=20, alpha=0.6)
axes[0].set_title('True Labels', fontsize=14)
plt.colorbar(scatter1, ax=axes[0])

scatter2 = axes[1].scatter(embeddings_2d[:, 0], embeddings_2d[:, 1], c=cluster_labels, cmap='tab10', s=20, alpha=0.6)
axes[1].set_title('K-Means Clusters', fontsize=14)
plt.colorbar(scatter2, ax=axes[1])

plt.tight_layout(); plt.show()

## 6. Cluster Analysis

In [ ]:
# Analyze cluster composition
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(true_labels, cluster_labels)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=[f'C{i}' for i in range(10)],
            yticklabels=class_names)
plt.xlabel('Cluster'); plt.ylabel('True Class')
plt.title('Confusion Matrix: True Classes vs Clusters')
plt.tight_layout(); plt.show()

In [ ]:
# Show sample images from each cluster
fig, axes = plt.subplots(2, 5, figsize=(15, 6))
for c in range(10):
    ax = axes[c // 5, c % 5]
    cluster_indices = np.where(cluster_labels == c)[0]
    if len(cluster_indices) > 0:
        idx = cluster_indices[0]
        img, _ = cifar_dataset[indices[idx]]
        img = img.permute(1, 2, 0).numpy()
        img = (img - img.min()) / (img.max() - img.min())
        ax.imshow(img)
        dominant_class = np.argmax(np.bincount(true_labels[cluster_indices]))
        ax.set_title(f'Cluster {c}\n({class_names[dominant_class]})')
    ax.axis('off')
plt.suptitle('Sample Images from Each Cluster', fontsize=14)
plt.tight_layout(); plt.show()

## 7. Hierarchical Clustering

In [ ]:
agg = AgglomerativeClustering(n_clusters=10, linkage='ward')
agg_labels = agg.fit_predict(embeddings_pca)

print("Hierarchical Clustering Results:")
print(f"  Silhouette Score: {silhouette_score(embeddings_pca, agg_labels):.4f}")
print(f"  ARI: {adjusted_rand_score(true_labels, agg_labels):.4f}")
print(f"  NMI: {normalized_mutual_info_score(true_labels, agg_labels):.4f}")

## 8. Model Comparison

In [ ]:
results = pd.DataFrame([
    {'Method': 'K-Means', 'Silhouette': silhouette_score(embeddings_pca, cluster_labels),
     'ARI': adjusted_rand_score(true_labels, cluster_labels), 'NMI': normalized_mutual_info_score(true_labels, cluster_labels)},
    {'Method': 'Hierarchical', 'Silhouette': silhouette_score(embeddings_pca, agg_labels),
     'ARI': adjusted_rand_score(true_labels, agg_labels), 'NMI': normalized_mutual_info_score(true_labels, agg_labels)}
])
print("\nMethod Comparison:")
print(results.to_string(index=False))

In [ ]:
# Bar chart comparison
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(results))
width = 0.25
ax.bar(x - width, results['Silhouette'], width, label='Silhouette', color='steelblue')
ax.bar(x, results['ARI'], width, label='ARI', color='coral')
ax.bar(x + width, results['NMI'], width, label='NMI', color='green')
ax.set_xticks(x); ax.set_xticklabels(results['Method'])
ax.set_ylabel('Score'); ax.set_title('Image Clustering Methods Comparison')
ax.legend(); ax.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.show()

## 9. Conclusion

### Key Findings:
- **Deep learning embeddings** (ResNet50) capture semantic image features effectively
- **UMAP** provides good visualization of high-dimensional image embeddings
- **K-Means** achieves reasonable clustering aligned with true categories
- Similar objects (vehicles, animals) tend to cluster together

### Applications:
- Image organization and retrieval
- Visual similarity search
- Content-based image classification
- Dataset exploration

In [ ]:
print("="*60)
print("IMAGE CLUSTERING WITH DEEP EMBEDDINGS - COMPLETE")
print("="*60)
print("\n✓ Loaded CIFAR-10 dataset")
print("✓ Extracted embeddings with ResNet50")
print("✓ Applied PCA and UMAP for dimensionality reduction")
print("✓ K-Means and Hierarchical clustering")
print("✓ Cluster analysis and visualization")
print("✓ Comprehensive evaluation metrics")